# VERA · PushT (client–server, minimalist)

Drives the **PushT** push-to-goal task against a running VERA policy server. Inference runs on the
server's GPU; this notebook only steps the env.

**Start the server first** (DFoT + Jacobian IDM are tiny — loads in seconds):
```bash
python -m vera.server.start_vera_server --embodiment pusht --port 8820 --vis-port 8821
```
Checkpoint paths come from the `VERA_PUSHT_*` env vars (see `vera/server/start_server_pusht.py`).
Live viewer (dream · flow · per-chunk play bar): **http://localhost:8821/**

In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import torch  # env stepping only; inference runs server-side
from pathlib import Path
import vera
print('vera:', vera.__file__)

vera: /path/to/VERA/vera/__init__.py


In [2]:
HOST, PORT, VIS_PORT = "127.0.0.1", 8820, 8821
ZARR_PATH = "/path/to/pusht/pusht_cchi_v7_replay.zarr"
OUTPUT_DIR = "outputs/vera_pusht_dfot_stack"
RENDER_SIZE = 252
HORIZON = 200
SUCCESS_THRESHOLD = 0.9
SEED = 42
# Walkthrough default: one start state, three repeats (server-side diffusion sampling varies
# per repeat). Set FRAME_INDICES = None to sample N_STATES (population SR).
FRAME_INDICES = [3664]
N_REPEATS = 3
N_STATES = 100

assert Path(ZARR_PATH).exists(), f"missing zarr: {ZARR_PATH}"
print("zarr OK")

zarr OK


In [3]:
# --- Connect: attach to the running server + print what it's serving ---
from vera.server.protocol.websocket_policy_client import WebsocketClientPolicy
client = WebsocketClientPolicy(host=HOST, port=PORT)
meta = client.get_server_metadata()
view_keys = list(meta['view_keys'])
context_frames = int(meta.get('context_frames', 9))
print('planner (DFoT):', meta.get('planner_model'))
print('IDM           :', meta.get('idm_model'))
print('views         :', view_keys, '| context_frames:', context_frames)
print('action_space  :', meta.get('action_space'), '| action_dim:', meta.get('action_dim'),
      '| gripper_dim_index:', meta.get('gripper_dim_index'))
print('H             :', meta.get('action_horizon'),
      '| control_hz:', round(1.0 / meta['control_dt'], 1) if meta.get('control_dt') else None)
print(f'live viewer   : http://localhost:{VIS_PORT}/')

planner (DFoT): default-wan
IDM           : pusht-idm
views         : ['image'] | context_frames: 2
action_space  : velocity | action_dim: 2 | gripper_dim_index: -1
H             : 2 | control_hz: 10.0
live viewer   : http://localhost:8821/


In [4]:
from vera.env_runner.pusht_runner import PushtRunnerCfg, PushTRunner
from vera.controller.run_mimicgen_eval import RemotePolicy   # planner-agnostic remote BasePolicy

runner_cfg = PushtRunnerCfg(
    env_name='pusht', n_repeat=1, num_env_train=1, num_env_eval=0,
    max_episode_steps=HORIZON, action_scale=1.0,   # runner integrates du + applies actions_vel_scale
    output_dir=OUTPUT_DIR, save_videos=True, save_trajectory=False, save_rrd=False, video_fps=5,
)
runner = PushTRunner(runner_cfg, device=torch.device('cpu'))  # setup_env() runs in __init__

# Single view -> view_keys=['image'], one width. RemotePolicy keeps a local context window +
# action queue and calls the server's chunked infer only on refill; the runner integrates the
# returned du into a position-velocity command.
remote = RemotePolicy(client, view_keys=view_keys,
                      view_widths=[RENDER_SIZE] * len(view_keys),
                      context_frames=context_frames, prompt=None)
print('runner + remote policy ready')

runner + remote policy ready


/path/to/venvs/client/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


### Success criterion

The env reward is **normalized coverage**: `reward = clip(coverage / 0.82, 0, 1)`, where 0.82 is
the env's `success_threshold` (see `vera/env_runner/env_wrappers/pusht_env.py`; upstream
`gym-pusht` defaults to 0.95). Two criteria appear in PushT reporting:

- **coverage &ge; 0.82** — `max_reward == 1.0` (the env's own success flag);
- **`max_reward` &ge; 0.9** — this notebook's `SUCCESS_THRESHOLD` default (coverage &ge; 0.738).

For reference, an n=200 uniform-state evaluation (100 start states uniform over the replay
buffer x 2 repeats, horizon 200) of the Wave-1 release configuration measured **SR = 34.0%**
at the coverage-0.82 criterion and **SR = 47.5%** at the `max_reward >= 0.9` criterion.
A confirmatory run of the current configuration (bundled renderer, 2 executed steps of a
3-frame plan) on the same protocol measured **SR = 58.5%** (95% CI 51.6-65.1%) at the
coverage-0.82 criterion and **SR = 65.5%** (95% CI 58.7-71.7%) at `max_reward >= 0.9`.

In [5]:
import random
import zarr

def _seed_everything(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

group   = zarr.open(ZARR_PATH, mode='r')
states  = group['data']['state']
n_total = states.shape[0]
if FRAME_INDICES is not None:
    indices = [int(i) for i in FRAME_INDICES]
else:
    indices = np.linspace(0, n_total - 1, N_STATES, dtype=int).tolist()
print(f'rolling out {len(indices)} state(s) x {N_REPEATS} repeat(s): '
      f'{indices if len(indices) <= 12 else str(indices[:12]) + " ..."}')

# The env reward is goal-region coverage normalized by the env success threshold:
# reward = clip(coverage / env_threshold, 0, 1) with env_threshold = 0.82. "max_reward" is
# the episode max of that normalized reward; implied coverage = min(max_reward, 1) * 0.82.
ENV_THRESHOLD = 0.82
rows = []
for fi in indices:
    for rep in range(1, N_REPEATS + 1):
        _seed_everything(SEED)
        reset_to_state = np.asarray(states[fi], dtype=np.float32)
        out = runner.run(remote, options={'reset_to_state': reset_to_state},
                         run_tag=f'state_{fi}_rep{rep}')
        mrm      = float(out.get('max_reward_mean', 0.0))
        coverage = min(mrm, 1.0) * ENV_THRESHOLD
        ret      = float(out['train_returns'][0]) if len(out['train_returns']) else 0.0
        success  = int(mrm >= SUCCESS_THRESHOLD)
        rows.append({'frame_idx': fi, 'repeat': rep, 'max_reward': mrm, 'coverage': coverage,
                     'return': ret, 'success': success, 'save_dir': out.get('save_dir')})
        print(f'  state {fi:>6d} rep{rep}:  max_reward={mrm:.4f}  coverage={coverage:.4f}  '
              f'return={ret:.4f}  success={success}')

sr  = 100.0 * np.mean([r['success'] for r in rows])
tp  = 100.0 * np.mean([r['max_reward'] for r in rows])
cov = 100.0 * np.mean([r['coverage'] for r in rows])
print('=' * 60)
print(f'PushT SR = {sr:.1f}%   (max_reward >= {SUCCESS_THRESHOLD}, n={len(rows)})')
print(f'mean max normalized reward = {tp:.1f}%   |   mean implied coverage = {cov:.1f}%')
print('=' * 60)

/path/to/venvs/client/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:135: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float64, actual type: float32
  logger.warn(


rolling out 1 state(s) x 3 repeat(s): [3664]


[PushTRunner] Rollout:   0%|          | 0/200 [00:00<?, ?it/s]

    chunk   1 (env step   1): dreaming + denoising on server…

    chunk   1 (env step   1): dream done in  2.0s → committing 2 actions (|a|=2.146)            


...

    chunk 100 (env step 199): dream done in  1.6s → committing 2 actions (|a|=3.602)            


[PushTRunner] Rollout: 100%|█████████▉| 199/200 [02:47<00:00,  1.17it/s]

[PushTRunner] Rollout: 100%|██████████| 200/200 [02:47<00:00,  1.19it/s]

[Step 198] pos_env=[295.42263794 349.14315796] v_cmd=[-2.7594714  3.5867643] pos_pred=[292.66316652 352.72992229]
[Step 199] pos_env=[288.5239563 358.1100769] v_cmd=[-5.2121887 -2.8489869] pos_pred=[283.31176758 355.26109004]

[PushTRunner] Rollout complete.
  Train returns: [136.7239]  (mean=136.724)
  Eval  returns: []  (mean=nan)


  state   3664 rep3:  max_reward=0.9964  coverage=0.8171  return=136.7239  success=1
PushT SR = 100.0%   (max_reward >= 0.9, n=3)
mean max normalized reward = 99.9%   |   mean implied coverage = 81.9%


In [6]:
import base64
from IPython.display import HTML, display

def _video_tag(mp4_path, fps=5, width=252):
    b64 = base64.b64encode(Path(mp4_path).read_bytes()).decode('ascii')
    rate = max(1.0, 60.0 / fps)   # play 200-step episodes at ~12x so they're watchable
    return (f'<video width="{width}" controls loop muted autoplay '
            f'onloadedmetadata="this.playbackRate={rate}">'
            f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

html = ["<table style='border-collapse:collapse'>",
        "<tr><th style='padding:6px'>state</th><th>repeat</th><th>max_reward</th><th>success</th>"
        "<th>policy view</th></tr>"]
for r in rows:
    vid = 'n/a'
    if r['save_dir']:
        mp4 = Path(r['save_dir']) / 'videos' / 'policy_env0.mp4'
        if mp4.exists():
            vid = _video_tag(mp4)
    color = '#0a0' if r['success'] else '#a00'
    html.append(
        f"<tr><td style='padding:6px'>{r['frame_idx']}</td>"
        f"<td style='padding:6px'>{r.get('repeat', 1)}</td>"
        f"<td style='color:{color};font-weight:600'>{r['max_reward']:.4f}</td>"
        f"<td style='color:{color};font-weight:600'>{r['success']}</td>"
        f"<td>{vid}</td></tr>")
html.append('</table>')
html.append(f"<p><b>SR = {sr:.1f}%</b> &nbsp; (max_reward &ge; {SUCCESS_THRESHOLD}, n = {len(rows)}) "
            f"&nbsp;|&nbsp; mean max normalized reward = {tp:.1f}% "
            f"&nbsp;|&nbsp; mean implied coverage = {cov:.1f}%</p>")
display(HTML(''.join(html)))

state,repeat,max_reward,success,policy view
3664,1,1.0000,1,
3664,2,1.0000,1,
3664,3,0.9964,1,


In [7]:
import urllib.request, json, time, base64, tempfile
import cv2, numpy as np, mediapy as media
from pathlib import Path
from IPython.display import HTML, display

def show_policy_vis(vis_host=HOST, vis_port=VIS_PORT, fps=10, max_width=1100):
    base = f"http://{vis_host}:{vis_port}"
    try:
        with urllib.request.urlopen(f"{base}/stats.json", timeout=5) as r:
            n = int(json.loads(r.read().decode()).get("video_len", 0))
    except Exception as e:
        print(f"[vis] server not reachable at {base} - is it up with --vis-port {vis_port}? ({e})"); return
    if n == 0:
        print("[vis] buffer empty - run the rollout cell first (watch the live viewer while it runs)."); return
    urllib.request.urlopen(f"{base}/video/pause", timeout=3)
    frames = []
    for i in range(n):
        urllib.request.urlopen(f"{base}/video/seek?pos={i}", timeout=3); time.sleep(0.05)
        with urllib.request.urlopen(f"{base}/policy.mjpg", timeout=5) as resp:
            buf = b""
            while True:
                chunk = resp.read(4096)
                if not chunk: break
                buf += chunk
                e = buf.find(b"\xff\xd9")
                if e != -1:
                    s = buf.find(b"\xff\xd8")
                    if s != -1:
                        img = cv2.imdecode(np.frombuffer(buf[s:e+2], np.uint8), cv2.IMREAD_COLOR)
                        if img is not None: frames.append(img)
                    break
    urllib.request.urlopen(f"{base}/video/play", timeout=3)
    if not frames:
        print("[vis] no frames grabbed."); return
    Hm = max(f.shape[0] for f in frames); Wm = max(f.shape[1] for f in frames)
    def _pad(f):
        h, w = f.shape[:2]
        if (h, w) == (Hm, Wm): return f
        out = np.zeros((Hm, Wm, 3), dtype=f.dtype); out[:h, :w] = f; return out
    rgb = np.stack([cv2.cvtColor(_pad(f), cv2.COLOR_BGR2RGB) for f in frames])
    mp4 = tempfile.mktemp(suffix=".mp4")
    media.write_video(mp4, rgb, fps=fps)
    b64 = base64.b64encode(Path(mp4).read_bytes()).decode()
    print(f"[vis] composite policy-vis: {len(rgb)} frames @ {Wm}x{Hm}")
    display(HTML(
        "<p><b>[ current | dream + tracks &#8594; dream | jacobian ]</b></p>"
        f'<video width="{min(Wm, max_width)}" controls loop autoplay muted>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

show_policy_vis()


[vis] composite policy-vis: 200 frames @ 630x342
